In [5]:
import warnings
warnings.filterwarnings('ignore')

In [36]:
# IMPORTS

%config Completer.use_jedi = False
import math
import shutil
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from PIL import Image
import seaborn as sns
import cv2
import random
import os
import imageio
import plotly.graph_objects as go
import plotly.express as px
import plotly.figure_factory as ff
from plotly.subplots import make_subplots
from collections import Counter

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.neighbors import LocalOutlierFactor
from sklearn.metrics import accuracy_score, recall_score, precision_score, classification_report, confusion_matrix
from sklearn.model_selection import RandomizedSearchCV, cross_val_score, RepeatedStratifiedKFold
from imblearn.over_sampling import SMOTE

import tensorflow as tf
import keras
from keras.models import Sequential
from keras.layers import Dense, Dropout, Activation, Flatten
from keras.layers import Conv2D, MaxPooling2D, GlobalAveragePooling2D, BatchNormalization, Dropout, Flatten, Dense, GlobalAvgPool2D
from keras.applications import resnet
from keras.applications.resnet import ResNet50
from keras.preprocessing.image import ImageDataGenerator
from keras.models import Sequential
from keras.preprocessing.image import load_img, img_to_array

In [9]:
# DATA

# If you are running from colab
"""

if 'google.colab' in str(get_ipython()):
    from google.colab import drive
    drive.mount('/content/drive')
    print('Running on CoLab')
    directory = r'/content/drive/Shareddrives/cancer-detection-model/MERGED The IQ-OTHNCCD lung cancer dataset'
else:
    # If you want to run local
    print('Not running on CoLab')
    directory ="D:\Volunteer_DS_2023\Omdena_Myanmar\Cancer_detection\Data"

categories = ['Bengin cases', 'Malignant cases', 'Normal cases']
"""

'\n\nif \'google.colab\' in str(get_ipython()):\n    from google.colab import drive\n    drive.mount(\'/content/drive\')\n    print(\'Running on CoLab\')\n    directory = r\'/content/drive/Shareddrives/cancer-detection-model/MERGED The IQ-OTHNCCD lung cancer dataset\'\nelse:\n    # If you want to run local\n    print(\'Not running on CoLab\')\n    directory ="D:\\Volunteer_DS_2023\\Omdena_Myanmar\\Cancer_detection\\Data"\n\ncategories = [\'Bengin cases\', \'Malignant cases\', \'Normal cases\']\n'

In [26]:
# count the number of images in the respective classes of the dataset (benign, malignant, normal)

ROOT_DIR = r"C:\Users\dorsa\PROJECTS\Machine-Learning-Projects\Lung Cancer Detection\data"
number_of_images = {}

for dir in os.listdir(ROOT_DIR):
    number_of_images[dir] = len(os.listdir(os.path.join(ROOT_DIR, dir)))

In [27]:
number_of_images.items()

dict_items([('Benign cases', 38), ('Malignant cases', 171), ('Normal cases', 127)])

In [33]:
def dataFolder(p, split):
    # create a train folder

    # if folder does NOT exist
    if not os.path.exists("./"+p):
        os.mkdir("./" + p)

        for dir in os.listdir(ROOT_DIR):
            os.makedirs("./"+p + "/" + dir)

            for img in np.random.choice(a = os.listdir(os.path.join(ROOT_DIR, dir)), 
                                        size = (math.floor(split * number_of_images[dir])-5),
                                        replace=False):
                O = os.path.join(ROOT_DIR, dir, img) # path
                D = os.path.join("./"+p, dir) # destination
                shutil.copy(O, D)
                os.remove(O)
    else:
        print(f"{p} folder exists")
            

In [29]:
dataFolder("train", 0.7)

train folder exists


In [35]:
dataFolder("test", 0.6)

Model Build

In [38]:
# CNN Model

model = Sequential()

# Add convolutional layers
model.add(Conv2D(filters=16, kernel_size=(3,3), activation = 'relu', input_shape = (224,224,3) ))

model.add(Conv2D(filters=36, kernel_size=(3,3), activation = 'relu' ))
model.add(MaxPooling2D(pool_size=(2,2)))

model.add(Conv2D(filters=64, kernel_size=(3,3), activation = 'relu' ))
model.add(MaxPooling2D(pool_size=(2,2)))

model.add(Conv2D(filters=128, kernel_size=(3,3), activation = 'relu' ))
model.add(MaxPooling2D(pool_size=(2,2)))

model.add(Dropout(rate = 0.25)) # retain 75% of data

model.add(Flatten()) # to prevent overfitting
model.add(Dense(64, activation = 'relu'))
model.add(Dropout(rate = 0.25))
model.add(Dense(units=1, activation='sigmoid'))

model.summary()

Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d_4 (Conv2D)           (None, 222, 222, 16)      448       
                                                                 
 conv2d_5 (Conv2D)           (None, 220, 220, 36)      5220      
                                                                 
 max_pooling2d_3 (MaxPoolin  (None, 110, 110, 36)      0         
 g2D)                                                            
                                                                 
 conv2d_6 (Conv2D)           (None, 108, 108, 64)      20800     
                                                                 
 max_pooling2d_4 (MaxPoolin  (None, 54, 54, 64)        0         
 g2D)                                                            
                                                                 
 conv2d_7 (Conv2D)           (None, 52, 52, 128)      